# Test de connexion à l'API Odoo (XML-RPC) → extraction des tickets Helpdesk

Identifiants lus dans l'environnement : `ODOO_URL`, `ODOO_DB`, `ODOO_USERNAME`, `ODOO_API_KEY` (voir `.env.example`).

In [ ]:
import os
import xmlrpc.client
import pandas as pd

In [ ]:
# =========================
# CONNEXION ODOO
# =========================
url = os.environ["ODOO_URL"]
db = os.environ["ODOO_DB"]
username = os.environ["ODOO_USERNAME"]
password = os.environ["ODOO_API_KEY"]

common = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/common")
uid = common.authenticate(db, username, password, {})

models = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/object")

print("Connexion OK | UID:", uid)

# =========================
# TICKETS (PAGINATION PROPRE)
# =========================
all_tickets = []
offset = 0
limit = 100
max_loops = 200  # sécurité
loop_count = 0

while True:
    loop_count += 1

    if loop_count > max_loops:
        print("Stop sécurité (boucle évitée)")
        break

    batch = models.execute_kw(
        db,
        uid,
        password,
        'helpdesk.ticket',
        'search_read',
        [[]],  # aucun filtre
        {
            'fields': [
                'id',
                'name',
                'create_date',
                'write_date',
                'stage_id',
                'user_id',
                'partner_id',
                'priority'
            ],
            'limit': limit,
            'offset': offset,
            'order': 'id'  # CRUCIAL pour éviter boucle infinie
        }
    )

    print(f"Offset: {offset} | Batch: {len(batch)}")

    if not batch:
        break

    all_tickets.extend(batch)
    offset += limit

df_tickets = pd.DataFrame(all_tickets)

# =========================
# NETTOYAGE
# =========================
if not df_tickets.empty:
    df_tickets['stage'] = df_tickets['stage_id'].apply(lambda x: x[1] if x else None)
    df_tickets['agent'] = df_tickets['user_id'].apply(lambda x: x[1] if x else None)
    df_tickets['client'] = df_tickets['partner_id'].apply(lambda x: x[1] if x else None)

print("Total tickets :", len(df_tickets))

# =========================
# USERS
# =========================
data_users = models.execute_kw(
    db, uid, password,
    'res.users', 'search_read', [[]],
    {'fields': ['id', 'name', 'login'], 'limit': 100}
)
df_users = pd.DataFrame(data_users)

# =========================
# CLIENTS
# =========================
data_clients = models.execute_kw(
    db, uid, password,
    'res.partner', 'search_read', [[]],
    {'fields': ['id', 'name'], 'limit': 100}
)
df_clients = pd.DataFrame(data_clients)

print("Users:", len(df_users), "| Clients:", len(df_clients))

In [ ]:
df_tickets.to_excel("tickets.xlsx", index=False)